In [2]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# ruff: noqa
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:72: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /gpfs/gsfs12/users/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_scatter/_scatter_cuda.so: undefined symbol: _ZN2at23SavedTensorDefaultHooks11set_tracingEb
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/torch_geometric/typing.py:99: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. S

In [4]:
method = "GAT"
name = "NCI_GAT_New"
data = "nci"
study = optuna.create_study(
    storage="sqlite:///{}.sqlite3".format(name),
    study_name=name,
    load_if_exists=True,
)
df = study.trials_dataframe()
df = df.dropna(subset=[i for i in df.columns if "values" in i])
tmp = df.loc[
    df[["values_0", "values_1", "values_2", "values_3"]]
    .max(axis=1)
    .sort_values(ascending=False)
    .index
]
params = tmp[tmp["params_gnn_layer"] == method].head().iloc[0]
print(params)
params = {
    i.replace("params_", ""): j
    for i, j in zip(pd.DataFrame(params).index, params)
    if "params" in i
}

[I 2025-04-02 18:52:22,342] Using an existing study with name 'NCI_GAT_New' instead of creating a new one.


number                                            137
values_0                                     0.658982
values_1                                     0.727157
values_2                                     0.730925
values_3                                     0.609317
datetime_start             2025-04-01 12:35:37.260208
datetime_complete          2025-04-01 13:33:22.498690
duration                       0 days 00:57:45.238482
params_T_max                                      NaN
params_activation                                gelu
params_amsgrad                                  False
params_dropout1                                   0.3
params_dropout2                                   0.3
params_epochs                                   546.0
params_gamma_step                                 NaN
params_gnn_layer                                  GAT
params_heads                                      5.0
params_hidden1                                 1028.0
params_hidden2              

In [5]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data(data)

load nci
unique drugs: 177
unique genes: 251
DTI unique genes:  251
Top 90% variable genes:  2383
Total:  2582
Final gene exp shape: (60, 2582)
Final drug Act shape: (1005, 60)


100%|██████████| 25/25 [00:06<00:00,  3.64it/s]


Done!


In [6]:
import math


def auto_convert_params(params, nan_replace=None):
    """パラメータの型を自動変換する関数

    Args:
        params (dict): 変換前のパラメータ辞書
        nan_replace: NaNの置換値（デフォルトはNone）

    Returns:
        dict: 型変換後のパラメータ辞書
    """
    converted = {}
    for k, v in params.items():
        # NaNの処理
        if isinstance(v, float) and math.isnan(v):
            converted[k] = nan_replace
        # 浮動小数点数 → 整数変換（例: 1028.0 → 1028）
        elif isinstance(v, float) and v.is_integer():
            converted[k] = int(v)
        # その他の値はそのまま保持
        else:
            converted[k] = v
    return converted

In [7]:
params = auto_convert_params(params, nan_replace=0)

In [8]:
params.update(
    {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "epochs": 1000,
    }
)

In [9]:
params

{'T_max': 0,
 'activation': 'gelu',
 'amsgrad': False,
 'dropout1': 0.3,
 'dropout2': 0.3,
 'epochs': 1,
 'gamma_step': 0,
 'gnn_layer': 'GAT',
 'heads': 5,
 'hidden1': 1028,
 'hidden2': 128,
 'hidden3': 128,
 'lr': 0.00042366451892503394,
 'momentum': 0,
 'nesterov': 0,
 'optimizer': 'AdamW',
 'patience_plateau': 0,
 'scheduler': None,
 'step_size': 0,
 'thresh_plateau': 0,
 'weight_decay': 4.315910204355739e-06,
 'n_drug': 1005,
 'n_cell': 60,
 'n_gene': 2582,
 'head': 1}

In [10]:
PATH = f"../{data}_data/"

In [12]:
k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

true_datas = pd.DataFrame()
predict_datas = pd.DataFrame()

for train_index, test_index in kfold.split(np.arange(pos_num)):
    sampler = RandomSampler(
        drugAct,
        train_index,
        test_index,
        null_mask,
        S_d,
        S_c,
        S_g,
        A_cg,
        A_dg,
        PATH,
    )
    (_, _, _, best_val_labels, best_val_prob, best_metrics, _, _, _) = drGAT.train(
        sampler, params=params, device=device, verbose=False
    )
    true_datas = pd.concat(
        [true_datas, pd.DataFrame(best_val_labels)],
        ignore_index=True,
        axis=1,
    )
    predict_datas = pd.concat(
        [predict_datas, pd.DataFrame(best_val_prob)],
        ignore_index=True,
        axis=1,
    )

true_datas.to_csv(f"true_{data}_{method}.csv")
predict_datas.to_csv(f"pred_{data}_{method}.csv")

Using device: cuda


100%|██████████| 1/1 [00:02<00:00,  2.42s/it]


Best model found at epoch 1
Using device: cuda


100%|██████████| 1/1 [00:02<00:00,  2.41s/it]


Best model found at epoch 1
Using device: cuda


100%|██████████| 1/1 [00:02<00:00,  2.46s/it]

Best model found at epoch 1


In [13]:
true_datas

,0,1,2
0,1,1,0
1,0,0,0
2,1,0,1
3,1,1,0
4,1,1,1
...,...,...,...
17953,0,0,0
17954,0,0,0
17955,0,0,1
17956,0,0,1


In [14]:
predict_datas

,0,1,2
0,0.426919,0.539105,0.483831
1,0.419410,0.558959,0.478712
2,0.438154,0.551906,0.467089
3,0.452869,0.548734,0.481774
4,0.446917,0.550849,0.462625
...,...,...,...
17953,0.455017,0.549036,0.448336
17954,0.454805,0.567098,0.494827
17955,0.450752,0.571229,0.455078
17956,0.448456,0.555345,0.496475
